# Enterprise AI Knowledge Agent Platform Demo

## Project Question

**Can a private enterprise AI agent platform answer questions across documents, SQL tables, and image notes with citations, confidence scoring, and hallucination checks?**


In [9]:
from pathlib import Path
import os
import sys

# =========================================================
# PROJECT ROOT SETUP
# =========================================================

# If notebook is inside notebooks/
ROOT = Path.cwd().parent

# Change working directory to project root
os.chdir(ROOT)

# Add project root to python path
sys.path.append(str(ROOT))

print("ROOT:", ROOT)
print("Current Working Directory:", Path.cwd())

# =========================================================
# VERIFY VECTOR STORE
# =========================================================

print("FAISS exists:", (ROOT / "vector_store/index.faiss").exists())
print("Metadata exists:", (ROOT / "vector_store/metadata.json").exists())

ROOT: /Users/yuzhang/projects/Machine_learning/06_enterprise_ai_knowledge_agent_platform
Current Working Directory: /Users/yuzhang/projects/Machine_learning/06_enterprise_ai_knowledge_agent_platform
FAISS exists: True
Metadata exists: True


In [10]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
print("Project root:", ROOT)

Project root: /Users/yuzhang/projects/Machine_learning/06_enterprise_ai_knowledge_agent_platform


## 1. Build vector store and SQL database

In [11]:
from rag.document_loader import load_documents, create_chunks
from rag.vector_store import build_vector_store
from tools.image_tool import load_image_notes
from tools.sql_tool import load_csv_to_sqlite

docs = load_documents(str(ROOT / "data/documents"))
doc_chunks = create_chunks(docs)

image_notes = load_image_notes(str(ROOT / "data/images"))
image_chunks = create_chunks(image_notes)

all_chunks = doc_chunks + image_chunks
build_vector_store(all_chunks, str(ROOT / "vector_store"))

load_csv_to_sqlite(
    csv_path=str(ROOT / "data/sql/project_portfolio_metrics.csv"),
    table_name="portfolio_metrics",
    db_path=str(ROOT / "data/sql/enterprise_agent.db")
)

print("Chunks indexed:", len(all_chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks indexed: 4


## 2. Ask the SQL agent

In [3]:
from agents.orchestrator import answer_question

result = answer_question(
    "Which portfolio projects have the highest priority and why?",
    index_dir=str(ROOT / "vector_store"),
    db_path=str(ROOT / "data/sql/enterprise_agent.db"),
    use_ollama=False,
)

print("Route:", result["route"])
print(result["answer"])
print("Confidence:", result["confidence"])
print("Hallucination:", result["hallucination"])

Route: sql_agent
Ollama disabled. Retrieved evidence summary:\n\nproject_portfolio_metrics.csv chunk sql:                     project                     domain                             primary_skill risk_level  portfolio_priority
        CAPSDAC Forecasting    Public Sector Education           Forecasting and data governance       High                   5
        Local RAG Assistant              Enterprise AI              RAG and local LLM deployment     Medium                   5
Medical Imaging 
Confidence: {'confidence_score': 0.75, 'confidence_label': 'high', 'reason': 'Average retrieval score=1.000; distinct sources=1.'}
Hallucination: {'risk_level': 'low', 'flags': []}


## 3. Ask the document RAG agent

In [12]:
result = answer_question(
    "What should healthcare AI governance include?",
    index_dir=str(ROOT / "vector_store"),
    db_path=str(ROOT / "data/sql/enterprise_agent.db"),
    use_ollama=False,
)

print("Route:", result["route"])
print(result["answer"])
print("Confidence:", result["confidence"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Route: document_rag_agent
Ollama disabled. Retrieved evidence summary:\n\nhealthcare_ai_governance.txt chunk 0: Healthcare AI Governance Notes

Clinical AI systems should support decision-making but should not replace clinician judgment.
A reliable enterprise AI platform should include data lineage, source citations, audit logs,
privacy controls, hallucination checks, and confidence scoring.

Important healthcare AI evaluation topics include model performance, fairness, calibration,
explainability, uncertai\n\nplatform_architecture_note.txt chunk 0: Image note: enterprise AI platform architecture diagram.

The diagram shows a user question routed through an agent router into document RAG, SQL analytics,
and image-note tools. Retrieved evidence is aggregated and passed to a local Ollama model.
The final answer includes citations, confidence score, and hallucination risk check.\n\npublic_sector_analytics_notes.txt chunk 0: Public Sector Analytics Notes

Public-sector analytics projects o

## 4. Run benchmark evaluation

In [13]:
from evaluation.evaluate_agent import evaluate_agent

results, summary = evaluate_agent(
    eval_path=str(ROOT / "evaluation/eval_questions.csv"),
    output_path=str(ROOT / "outputs/tables/agent_evaluation_results.csv")
)

display(summary)
display(results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   hit_rate  average_confidence  n_questions
0       1.0               0.565            5


,hit_rate,average_confidence,n_questions
0,1.0,0.565,5


,question,expected_source_keyword,route,hit_in_sources,confidence_score,confidence_label,hallucination_risk,sources
0,What should healthcare AI governance include?,healthcare_ai_governance,document_rag_agent,True,0.558,medium,low,"healthcare_ai_governance.txt, platform_archite..."
1,Why is de-identification important for public-...,public_sector_analytics,document_rag_agent,True,0.518,medium,low,"public_sector_analytics_notes.txt, platform_ar..."
2,What is an enterprise RAG architecture?,enterprise_rag_architecture,document_rag_agent,True,0.518,medium,low,"enterprise_rag_architecture.txt, platform_arch..."
3,Which portfolio projects have highest priority?,project_portfolio_metrics,sql_agent,True,0.750,high,low,project_portfolio_metrics.csv
4,What does the platform architecture diagram show?,platform_architecture_note,image_agent,True,0.481,medium,low,"platform_architecture_note.txt, enterprise_rag..."
